# Modelagem em Python – Tempo de volta em Mônaco (F1 23): carro ou setup?
Versão **Python** (Google Colab, ambiente Python padrão) do `modelagem_colab.ipynb` em R. **Mesmo desenho**: mesma limpeza, S10 fora do treino, teste por sessão, validação cruzada agrupada por bloco (carro × sessão × setup), trilha A (previsão/deploy) × trilha B (explicativa com telemetria).

Usa os **dois arquivos** do logger `capturar_telemetria.py`: `voltas.csv` (1 linha por volta) e `amostras.csv` (~4.300 linhas por volta).

> **Regra da disciplina:** IA permitida para dúvidas de sintaxe, **mas o grupo precisa explicar cada decisão**. Antes de cada código há um **Por quê** e **Perguntas para estar pronto**.

> **Dados sintéticos:** existe um gerador de demonstração **desligado por padrão** (`USAR_DADOS_SINTETICOS = False`). Números sintéticos são **FALSOS** e nunca devem ir para a apresentação.

## 0. Bibliotecas
**Por quê:** o Colab já traz pandas, scikit-learn, xgboost, plotly e matplotlib; instalamos só o que faltar (`shap`, `lightgbm`). SHAP e LightGBM são **opcionais**: se não instalarem, o notebook segue sem eles.

**Perguntas:** *Por que scikit-learn?* → mesma interface (`Pipeline` + `GridSearchCV`) para todos os modelos, comparação justa com as mesmas folds e métricas – o equivalente ao tidymodels da aula.

In [ ]:
import importlib, subprocess, sys
for mod, pkg in [("xgboost", "xgboost"), ("shap", "shap"), ("lightgbm", "lightgbm"), ("plotly", "plotly")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px, plotly.graph_objects as go
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_validate, cross_val_predict, train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import joblib
warnings.filterwarnings("ignore")
def tem(mod):
    try: importlib.import_module(mod); return True
    except Exception: return False
TEM_XGB, TEM_SHAP, TEM_LGBM = tem("xgboost"), tem("shap"), tem("lightgbm")
if TEM_XGB: from xgboost import XGBRegressor
if TEM_LGBM: from lightgbm import LGBMRegressor
SEMENTE = 2026
np.random.seed(SEMENTE)
print("xgboost:", TEM_XGB, "| shap:", TEM_SHAP, "| lightgbm:", TEM_LGBM)

## 1. Onde estão os dados
Três formas (escolha em `MODO`):
- `"upload"` – abre a janela de upload do Colab (`google.colab.files`);
- `"drive"` – monta o Google Drive e lê de `PASTA_DRIVE` (melhor para o `amostras.csv`, que é grande);
- `"url"` – links diretos em `URL_VOLTAS` / `URL_AMOSTRAS`.
Se os arquivos já estiverem em `/content` (barra lateral), nada precisa ser feito.

**Por quê:** o `amostras.csv` pode ter centenas de MB; Drive evita reenviar a cada sessão. As células específicas do Colab ficam em `try/except` para o notebook também rodar fora dele.

In [ ]:
USAR_DADOS_SINTETICOS = os.environ.get("F1_SINTETICO", "False") == "True"   # troque para True SÓ para testar (dados FALSOS)
MODO = "pasta"            # "pasta" | "upload" | "drive" | "url"
PASTA = "/content" if os.path.isdir("/content") else os.getcwd()
PASTA_DRIVE = "/content/drive/MyDrive/f1_monaco"
URL_VOLTAS, URL_AMOSTRAS = "", ""

if not USAR_DADOS_SINTETICOS:
    try:
        if MODO == "upload":
            from google.colab import files
            for nome in files.upload():
                print("recebido:", nome)
        elif MODO == "drive":
            from google.colab import drive
            drive.mount("/content/drive"); PASTA = PASTA_DRIVE
    except ImportError:
        print("Fora do Colab: lendo arquivos de", PASTA)
print("Pasta:", PASTA)

### 1a. (Opcional) Gerador SINTÉTICO – desligado por padrão
Mesmo estilo do `dashboard/gerar_demo.py`: traçado de Mônaco do GeoJSON público *bacinger/f1-circuits* (mc-1929), perfil de velocidade calculado pela curvatura (limite de aderência + aceleração/frenagem), e as **mesmas colunas** do `capturar_telemetria.py`.
Plano igual ao real: Ferrari sessões 1–2, Alfa Romeo 3–4, S10 nas sessões 5 (Ferrari) e 6 (Alfa); S0 no início e no fim de cada sessão; 2 voltas de aquecimento + 10 por setup. **Efeitos de setup inventados.**

In [ ]:
MONACO_LONLAT = [[7.427191, 43.739404], [7.427171, 43.739494], [7.427199, 43.739575], [7.427271, 43.739686], [7.427339, 43.739764], [7.427551, 43.739958], [7.428628, 43.741015], [7.428703, 43.741091], [7.428814, 43.741114], [7.428949, 43.741086], [7.429043, 43.74103], [7.429111, 43.740942], [7.429125, 43.740843], [7.429128, 43.740761], [7.429139, 43.740694], [7.429195, 43.740614], [7.42926, 43.740553], [7.429381, 43.74043], [7.429463, 43.740321], [7.429495, 43.740255], [7.429536, 43.740216], [7.429601, 43.740203], [7.429676, 43.740219], [7.429732, 43.740261], [7.429749, 43.740321], [7.429712, 43.740372], [7.429617, 43.740422], [7.429336, 43.74068], [7.429296, 43.740742], [7.429295, 43.740802], [7.429317, 43.740848], [7.429347, 43.740891], [7.429479, 43.74094], [7.429683, 43.740978], [7.429794, 43.741008], [7.429908, 43.741041], [7.430066, 43.741081], [7.430238, 43.74109], [7.430329, 43.741057], [7.430363, 43.740955], [7.430331, 43.740359], [7.430204, 43.739746], [7.430072, 43.739356], [7.429925, 43.739078], [7.429673, 43.73876], [7.429383, 43.738491], [7.428963, 43.73816], [7.428362, 43.737894], [7.428004, 43.737767], [7.427314, 43.737534], [7.42685, 43.737418], [7.426165, 43.737301], [7.425724, 43.737251], [7.42549, 43.737219], [7.42545, 43.73717], [7.425447, 43.737127], [7.425433, 43.737077], [7.425403, 43.737028], [7.425304, 43.737003], [7.425164, 43.736986], [7.425009, 43.73705], [7.424924, 43.737054], [7.42369, 43.736937], [7.423077, 43.736873], [7.422513, 43.736809], [7.422357, 43.736792], [7.422253, 43.736779], [7.422178, 43.736723], [7.42206, 43.736555], [7.42196, 43.736392], [7.42188, 43.736229], [7.421845, 43.736113], [7.421808, 43.735911], [7.421805, 43.735565], [7.42181, 43.735479], [7.421845, 43.73542], [7.421915, 43.735363], [7.42207, 43.735281], [7.422174, 43.735212], [7.422219, 43.735138], [7.422237, 43.735031], [7.422413, 43.734295], [7.422458, 43.734166], [7.422472, 43.734125], [7.422468, 43.73407], [7.422423, 43.734026], [7.422324, 43.733976], [7.422225, 43.733941], [7.422202, 43.733891], [7.422221, 43.733786], [7.422334, 43.733547], [7.422437, 43.733367], [7.422542, 43.733225], [7.422722, 43.733039], [7.422948, 43.732874], [7.423114, 43.732805], [7.4233, 43.732742], [7.423393, 43.732712], [7.42344, 43.732674], [7.423494, 43.732557], [7.423468, 43.732495], [7.423384, 43.732459], [7.423011, 43.732388], [7.422803, 43.732364], [7.422666, 43.732362], [7.422544, 43.732375], [7.422467, 43.732417], [7.42242, 43.732452], [7.422418, 43.732534], [7.4224, 43.7326], [7.422364, 43.732681], [7.422154, 43.732868], [7.422, 43.733047], [7.42188, 43.733181], [7.421753, 43.733435], [7.421472, 43.734142], [7.421395, 43.73437], [7.421341, 43.734614], [7.421271, 43.73497], [7.421216, 43.735433], [7.421216, 43.735735], [7.421234, 43.735966], [7.421269, 43.736206], [7.421331, 43.736373], [7.421401, 43.736523], [7.421435, 43.736615], [7.421452, 43.736696], [7.421444, 43.736819], [7.421441, 43.736895], [7.421475, 43.73694], [7.421562, 43.73697], [7.421774, 43.737007], [7.421963, 43.737013], [7.422222, 43.737036], [7.422583, 43.737069], [7.422903, 43.73711], [7.423717, 43.737283], [7.424172, 43.737373], [7.424589, 43.737422], [7.424875, 43.737439], [7.425098, 43.737497], [7.425743, 43.737718], [7.426026, 43.737802], [7.426439, 43.737869], [7.426784, 43.737904], [7.427113, 43.737944], [7.427298, 43.737991], [7.427523, 43.738093], [7.427718, 43.73824], [7.427828, 43.738364], [7.427877, 43.738494], [7.427869, 43.73863], [7.427837, 43.738758], [7.427767, 43.738899], [7.427674, 43.739033], [7.427528, 43.739115], [7.427388, 43.73921], [7.42732, 43.739253], [7.427257, 43.739299], [7.427191, 43.739404]]   # bacinger/f1-circuits mc-1929

def gerar_sintetico(seed=23, pts_volta=600):
    rng = np.random.default_rng(seed)
    c = np.array(MONACO_LONLAT, dtype=float)
    if np.allclose(c[0], c[-1]): c = c[:-1]
    lon0, lat0 = c[0]
    P = np.c_[(c[:, 0] - lon0) * 111320 * np.cos(np.radians(lat0)), (c[:, 1] - lat0) * 110540]
    Q = np.vstack([P, P[:1]]); seg = np.hypot(*np.diff(Q, axis=0).T); s = np.r_[0, np.cumsum(seg)]
    si = np.linspace(0, s[-1], pts_volta, endpoint=False)
    x, z = np.interp(si, s, Q[:, 0]) * 3337 / s[-1], np.interp(si, s, Q[:, 1]) * 3337 / s[-1]
    N, ds = pts_volta, 3337 / pts_volta
    head = np.unwrap(np.arctan2(np.roll(z, -1) - z, np.roll(x, -1) - x))
    k = np.abs(np.diff(np.r_[head, head[0] + (head[-1] - head[0]) * 0 + head[0] - head[0]]))
    k = np.abs(np.gradient(head)) / ds
    k = np.convolve(np.r_[k[-5:], k, k[:5]], np.ones(7) / 7, "same")[5:-5]
    def perfil(alat=30, vtop=80, acc=9, brk=38):
        v = np.minimum(vtop, np.sqrt(alat / np.maximum(k, 1e-6)))
        for _ in range(2):
            for i in range(1, 2 * N): j, p = i % N, (i - 1) % N; v[j] = min(v[j], np.sqrt(v[p] ** 2 + 2 * acc * ds))
            for i in range(2 * N, 0, -1): j, p = i % N, (i + 1) % N; v[j] = min(v[j], np.sqrt(v[p] ** 2 + 2 * brk * ds))
        return v
    base = perfil()
    sh = (int(np.argmin(base)) - int(0.41 * N)) % N             # grampo em ~41% da volta
    x, z, k, base = (np.roll(a, -sh) for a in (x, z, k, base))
    t_base = (ds / base).sum()
    S0 = dict(asa_dianteira=50, asa_traseira=50, dif_com_acel=50, dif_sem_acel=53, cambagem_diant=-2.5, cambagem_tras=-1.0,
              convergencia_diant=0.05, convergencia_tras=0.25, suspensao_diant=41, suspensao_tras=13, barra_diant=7, barra_tras=5,
              altura_diant=32, altura_tras=36, freio_pressao=100, freio_balanco=53, pneu_diant_esq_psi=22.5, pneu_diant_dir_psi=22.5,
              pneu_tras_esq_psi=20.0, pneu_tras_dir_psi=20.0)
    MUDA = {"S1": dict(dif_com_acel=80), "S2": dict(dif_sem_acel=30), "S3": dict(cambagem_diant=-3.5, cambagem_tras=-2.0),
            "S4": dict(convergencia_tras=0.10), "S5": dict(barra_diant=15), "S6": dict(barra_tras=12),
            "S7": dict(altura_diant=25, altura_tras=45), "S8": dict(pneu_tras_esq_psi=22.0, pneu_tras_dir_psi=22.0),
            "S9": dict(asa_traseira=45),
            "S10": dict(asa_traseira=47, dif_sem_acel=50, convergencia_diant=0.08, convergencia_tras=0.20, suspensao_diant=12,
                        suspensao_tras=7, barra_diant=5, barra_tras=2, altura_diant=36, altura_tras=42, freio_balanco=54,
                        pneu_diant_esq_psi=22.2, pneu_diant_dir_psi=22.2)}
    EFEITO = dict(S0=0, S1=0.25, S2=-0.1, S3=-0.3, S4=0.4, S5=0.15, S6=0.3, S7=0.6, S8=0.2, S9=-0.2, S10=-0.25)  # s, INVENTADO
    PLANO = {1: ("ferrari", ["S0", "S1", "S2", "S3", "S4", "S0"]), 2: ("ferrari", ["S0", "S5", "S6", "S7", "S8", "S9", "S0"]),
             3: ("alfaromeo", ["S0", "S1", "S2", "S3", "S4", "S0"]), 4: ("alfaromeo", ["S0", "S5", "S6", "S7", "S8", "S9", "S0"]),
             5: ("ferrari", ["S0", "S10", "S0"]), 6: ("alfaromeo", ["S0", "S10", "S0"])}
    voltas, amostras = [], []
    t0 = pd.Timestamp("2026-10-25 14:00")
    for ses, (carro, plano) in PLANO.items():
        uid, nv, drift = f"sint{ses}", 0, 0.0
        for st in plano:
            p = {**S0, **MUDA.get(st, {})}
            for kk in range(12):
                nv += 1; drift -= 0.012
                alvo = t_base + (0.6 if carro == "alfaromeo" else 0) + EFEITO[st] + drift + rng.normal(0, 0.3) + (3 if kk < 2 else 0)
                ruido = 1 + np.convolve(rng.normal(0, 0.02, N), np.ones(25) / 25, "same")
                v = base * ruido; v = v * (ds / v).sum() / alvo
                dt = ds / v; t_ms = np.cumsum(dt) * 1000; tempo = int(round(t_ms[-1]))
                dv = np.roll(v, -1) - v
                vk = v * 3.6
                amostras.append(pd.DataFrame(dict(
                    session_uid=uid, carro=carro, setup_id=st, sessao=ses, volta=nv, lap_distance=np.arange(N) * ds,
                    tempo_volta_atual_ms=np.round(t_ms - dt[0] * 1000), world_x=x, world_z=z, velocidade=np.round(vk),
                    acelerador=np.where(dv >= 0, 1.0, 0.0), freio=np.clip(-dv / 1.5, 0, 1).round(2),
                    marcha=np.clip(1 + vk // 40, 1, 8), g_lateral=(v ** 2 * k / 9.81).round(2),
                    temp_pneu_sup_te=95 + rng.normal(0, 2), temp_pneu_sup_td=95 + rng.normal(0, 2),
                    temp_pneu_sup_de=98 + rng.normal(0, 2), temp_pneu_sup_dd=98 + rng.normal(0, 2))))
                s1, s2 = int(t_ms[int(N * .3)]), int(t_ms[int(N * .7)] - t_ms[int(N * .3)])
                voltas.append(dict(session_uid=uid, data_hora=t0 + pd.Timedelta(seconds=80 * len(voltas)), carro=carro,
                    setup_id=st, sessao=ses, pista_id=5, pista="Monaco", equipe_id=1 if carro == "ferrari" else 9, volta=nv,
                    tempo_volta_ms=tempo, setor1_ms=s1, setor2_ms=s2, setor3_ms=tempo - s1 - s2,
                    valida=0 if kk == 1 or rng.random() < 0.06 else 1, volta_saida=int(kk == 0), entrou_box=0,
                    flashbacks=int(rng.random() < 0.03), vel_max_kmh=float(vk.max()), **p))
    return pd.DataFrame(voltas), pd.concat(amostras, ignore_index=True)

### 1b. Leitura
O `amostras.csv` é lido com **`usecols`** (só as colunas usadas) e tipos `float32`, para caber na memória do Colab.

In [ ]:
COLS_AMOSTRA = ["session_uid", "carro", "setup_id", "sessao", "volta", "lap_distance", "tempo_volta_atual_ms",
                "world_x", "world_z", "velocidade", "acelerador", "freio", "marcha", "g_lateral",
                "temp_pneu_sup_te", "temp_pneu_sup_td", "temp_pneu_sup_de", "temp_pneu_sup_dd"]
def achar(nomes, url):
    if url: return url
    for n in nomes:
        f = os.path.join(PASTA, n)
        if os.path.exists(f): return f
    return None

if USAR_DADOS_SINTETICOS:
    bruto, amostras = gerar_sintetico()
    print("ATENÇÃO: DADOS SINTÉTICOS (FALSOS) – só para testar o código!")
else:
    fv = achar(["voltas.csv", "voltas_limpas.csv"], URL_VOLTAS)
    if fv is None: raise FileNotFoundError(f"Envie voltas.csv (ou voltas_limpas.csv) para {PASTA}")
    bruto = pd.read_csv(fv, dtype={"session_uid": str})
    fa = achar(["amostras.csv"], URL_AMOSTRAS)
    amostras = None
    if fa:
        amostras = pd.read_csv(fa, usecols=lambda c: c in COLS_AMOSTRA, dtype={"session_uid": str})
        for c in amostras.select_dtypes("float64"): amostras[c] = amostras[c].astype("float32")
TEM_AMOSTRAS = amostras is not None and len(amostras) > 0
print("voltas:", len(bruto), "| amostras:", len(amostras) if TEM_AMOSTRAS else "não encontrado (telemetria será pulada)")

## 2. Limpeza (mesmas regras do `ler_dados` e do notebook R)
Válida pelo jogo, sem volta de saída, sem box, sem flashback, Mônaco (`pista_id == 5`) e corte acima de **107% da mediana** de cada carro × setup. As amostras ficam só das voltas limpas (chave `session_uid + carro + setup_id + sessao + volta`).

Também criamos: `fase_s0` (S0 do **início** ou do **fim** da sessão), `bloco` (carro × sessão × setup – unidade dos grupos da validação) e `volta_stint` (nº da volta dentro do bloco).

**Por quê:** o modelo deve aprender o setup, não os erros. Regras definidas antes de modelar, iguais para todos.

**Perguntas:** *Vocês descartaram dados para melhorar o resultado?* → não, as regras são objetivas e prévias. *Por que 107%?* → regra da própria F1.

In [ ]:
SETUPS = [f"S{i}" for i in range(11)]
df = bruto.copy()
for col, ok in [("valida", 1), ("volta_saida", 0), ("entrou_box", 0), ("flashbacks", 0), ("pista_id", 5)]:
    if col in df: df = df[df[col] == ok]
df["tempo_s"] = df["tempo_volta_ms"] / 1000
df["setup_id"] = df["setup_id"].astype(str)
df = df[df["setup_id"].isin(SETUPS) & (df["tempo_s"] > 0)]
med = df.groupby(["carro", "setup_id"])["tempo_s"].transform("median")
df = df[df["tempo_s"] <= 1.07 * med].copy()
df["session_uid"] = df["session_uid"].astype(str)

ordem = "data_hora" if "data_hora" in df else "volta"
df = df.sort_values(["sessao", ordem]).reset_index(drop=True)
def fase(g):
    nao_s0 = np.where(g["setup_id"].values != "S0")[0]
    primeira = nao_s0[0] if len(nao_s0) else len(g)
    pos = np.arange(len(g))
    return pd.Series(np.where(g["setup_id"] != "S0", "", np.where(pos < primeira, "inicio", "fim")), index=g.index)
df["fase_s0"] = df.groupby("sessao", group_keys=False).apply(fase)
df["bloco"] = df["carro"] + "_" + df["sessao"].astype(str) + "_" + df["setup_id"] + "_" + df["fase_s0"]
df["volta_stint"] = df.groupby("bloco").cumcount() + 1
df["pneu_diant"] = df[[c for c in ["pneu_diant_esq_psi", "pneu_diant_dir_psi"] if c in df]].mean(axis=1)
df["pneu_tras"] = df[[c for c in ["pneu_tras_esq_psi", "pneu_tras_dir_psi"] if c in df]].mean(axis=1)
limpas = df
print("Voltas limpas:", len(limpas), "de", len(bruto))

CHAVES = ["session_uid", "carro", "setup_id", "sessao", "volta"]
if TEM_AMOSTRAS:
    amostras["session_uid"] = amostras["session_uid"].astype(str); amostras["setup_id"] = amostras["setup_id"].astype(str)
    am = amostras[amostras["lap_distance"] >= 0].merge(limpas[CHAVES], on=CHAVES, how="inner")
    del amostras
    print("Amostras de voltas limpas:", len(am))

## 3. EDA – voltas
### 3a. Contagem, estatísticas e boxplot interativo
**Por quê:** confirmar ~10 voltas por carro × setup (equilíbrio) e ver se as diferenças entre setups são maiores que a dispersão dentro deles.

**Perguntas:** *O dataset é balanceado?* · *Qual o desvio-padrão típico de uma volta?*

In [ ]:
resumo = (limpas.groupby(["carro", "setup_id"])["tempo_s"].agg(n="count", melhor="min", media="mean", dp="std")
          .reset_index())
resumo["setup_id"] = pd.Categorical(resumo["setup_id"], SETUPS, ordered=True)
display(resumo.sort_values(["carro", "setup_id"]).round(3))
fig = px.box(limpas, x="setup_id", y="tempo_s", color="carro", category_orders={"setup_id": SETUPS}, points="all",
             title="Tempo de volta por setup e carro (Mônaco)", labels={"tempo_s": "Tempo (s)", "setup_id": "Setup"})
fig.show()

### 3b. Ruído do piloto: S0 no início × S0 no fim
O mesmo setup (S0) no começo e no fim de cada sessão. A diferença mede quanto **o piloto e a pista** mudam.

**Por quê:** é a **régua** do projeto: diferença entre setups menor que esse ruído = empate. Nos trabalhos parecidos (GT3/Assetto Corsa) quem pilota é a IA; aqui é humano, então precisamos medir.

**Perguntas:** *Como sabem que a melhora não foi só o piloto aquecendo?*

In [ ]:
drift = (limpas[limpas.setup_id == "S0"].groupby(["carro", "sessao", "fase_s0"])["tempo_s"].mean()
         .unstack("fase_s0").reset_index())
drift["drift_s"] = drift.get("fim") - drift.get("inicio")
display(drift.round(3))
RUIDO = float(drift["drift_s"].abs().mean())
if not np.isfinite(RUIDO): RUIDO = float(limpas.loc[limpas.setup_id == "S0", "tempo_s"].std())
print(f"Ruído do piloto (média |S0 fim − S0 início|): {RUIDO:.3f} s")

### 3c. Correlações (S0–S9)
**Por quê:** relações lineares simples e colunas constantes (sem variação não ensinam nada). Como S1–S9 mudam um fator por vez, parâmetros não deveriam andar juntos (exceção: S3 muda as duas cambagens; S7 as duas alturas).

**Perguntas:** *Correlação prova causa?* → não; o desenho um-fator-por-vez é que dá força à interpretação.

In [ ]:
PARAMS = [c for c in ["asa_dianteira", "asa_traseira", "dif_com_acel", "dif_sem_acel", "cambagem_diant", "cambagem_tras",
                      "convergencia_diant", "convergencia_tras", "suspensao_diant", "suspensao_tras", "barra_diant",
                      "barra_tras", "altura_diant", "altura_tras", "freio_pressao", "freio_balanco", "pneu_diant", "pneu_tras"]
          if c in limpas]
base_cor = limpas[limpas.setup_id != "S10"][PARAMS + ["tempo_s"]]
varia = [c for c in base_cor if base_cor[c].std() > 0]
print("Sem variação (o modelo ignora):", [c for c in PARAMS if c not in varia])
corr = base_cor[varia].corr()
px.imshow(corr, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, text_auto=".2f",
          title="Correlação entre parâmetros e tempo (S0–S9)").show()

## 4. EDA – telemetria (`amostras.csv`)
Ideias usadas:
1. **Mapa de Mônaco colorido pela velocidade** (a partir de `world_x`/`world_z`);
2. **Mapa dos pontos de frenagem** (onde o piloto começa a frear em cada volta);
3. **Velocidade × distância** de cada setup sobre o S0;
4. **Curvas lentas detectadas automaticamente** (mínimos locais < 150 km/h) e velocidade mínima em cada uma;
5. **% acelerador cheio / % freando**, **pneus**;
6. **Ganho em 20 trechos** vs. S0 (onde o setup ganha/perde);
7. **Consistência** (dp da velocidade entre voltas no mesmo ponto).

Tudo numa **grade de 10 m** para comparar voltas ponto a ponto.

**Por quê:** o boxplot diz *se* um setup é mais rápido; a telemetria diz *onde* e *como*.

**Perguntas:** *Onde o melhor setup ganha tempo?* · *Como acharam as curvas?* (nos próprios dados, não digitadas) · *Ganho no trecho é causa?* (associação; o piloto também varia).

In [ ]:
if TEM_AMOSTRAS:
    PASSO = 10
    am["dist_bin"] = (am["lap_distance"] // PASSO * PASSO).astype(int)
    perfil_volta = am.groupby(CHAVES + ["dist_bin"], observed=True).agg(vel=("velocidade", "mean")).reset_index()
    perfil = (perfil_volta.groupby(["carro", "setup_id", "dist_bin"])["vel"].agg(vel_media="mean", vel_dp="std").reset_index())
    # 1) mapa da pista colorido pela velocidade média do S0
    if {"world_x", "world_z"} <= set(am.columns):
        mapa = (am[am.setup_id == "S0"].groupby("dist_bin")
                  .agg(x=("world_x", "median"), z=("world_z", "median"), vel=("velocidade", "mean")).reset_index())
        fig = px.scatter(mapa, x="x", y="z", color="vel", color_continuous_scale="Turbo",
                         title="Mônaco – velocidade média no S0 (km/h)", hover_data=["dist_bin"])
        fig.update_traces(marker_size=7); fig.update_yaxes(scaleanchor="x"); fig.show()
    else:
        print("world_x/world_z ausentes – mapa pulado")

In [ ]:
if TEM_AMOSTRAS and {"world_x", "world_z", "freio"} <= set(am.columns):
    # 2) pontos de frenagem: primeira amostra com freio > 0,5 depois de uma amostra sem freio
    a = am.sort_values(CHAVES + ["lap_distance"])
    freando = a["freio"].values > 0.5
    inicio = freando & ~np.r_[False, freando[:-1]]
    pts = a.loc[inicio, ["carro", "setup_id", "world_x", "world_z", "lap_distance"]]
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    for i, (carro, g) in enumerate(pts.groupby("carro")):
        ax[i].plot(mapa.x, mapa.z, color="lightgrey", lw=6, zorder=0)
        for st, cor in [("S0", "black"), ("S10", "red")]:
            gg = g[g.setup_id == st]
            ax[i].scatter(gg.world_x, gg.world_z, s=8, alpha=.4, color=cor, label=st)
        ax[i].set_title(f"Pontos de frenagem – {carro} (S0 × S10)"); ax[i].set_aspect("equal"); ax[i].legend()
    plt.show()

In [ ]:
if TEM_AMOSTRAS:
    # 3) velocidade × distância: cada setup sobre o S0 (clique na legenda para ligar/desligar)
    for carro, g in perfil.groupby("carro"):
        fig = go.Figure()
        for st in SETUPS:
            gg = g[g.setup_id == st]
            if len(gg) == 0: continue
            fig.add_trace(go.Scatter(x=gg.dist_bin, y=gg.vel_media, name=st, mode="lines",
                                     line=dict(width=3 if st == "S0" else 1.2, color="black" if st == "S0" else None),
                                     visible=True if st in ("S0", "S10", "S9") else "legendonly"))
        fig.update_layout(title=f"Velocidade × distância – {carro} (S0 em preto)", xaxis_title="Distância (m)", yaxis_title="km/h")
        fig.show()

In [ ]:
if TEM_AMOSTRAS:
    # 4) curvas lentas = mínimos locais do S0 (janela ±100 m), abaixo de 150 km/h
    s0g = perfil[perfil.setup_id == "S0"].groupby("dist_bin")["vel_media"].mean().sort_index()
    suave = s0g.rolling(5, center=True, min_periods=1).mean()
    janela = suave.rolling(21, center=True, min_periods=1).min()
    mins = list(suave.index[(suave < 150) & (suave == janela)])
    mins = [m for i, m in enumerate(mins) if i == 0 or m - mins[i - 1] > 100]
    ZONAS = pd.DataFrame({"zona": [f"curva_{i+1:02d}_{m}m" for i, m in enumerate(mins)], "ini": [m - 40 for m in mins], "fim": [m + 40 for m in mins]})
    display(ZONAS)

    # agregados por volta (viram features da trilha B)
    temp_cols = [c for c in ["temp_pneu_sup_te", "temp_pneu_sup_td", "temp_pneu_sup_de", "temp_pneu_sup_dd"] if c in am]
    g = am.groupby(CHAVES)
    tele = pd.DataFrame({"tel_vel_max": g["velocidade"].max(), "tel_vel_media": g["velocidade"].mean(),
                         "tel_pct_acel_cheio": g["acelerador"].apply(lambda s: (s >= .95).mean() * 100),
                         "tel_pct_freando": g["freio"].apply(lambda s: (s > .05).mean() * 100)})
    if temp_cols: tele["tel_temp_pneu"] = g[temp_cols].mean().mean(axis=1)
    if "g_lateral" in am: tele["tel_g_lat"] = g["g_lateral"].apply(lambda s: s.abs().mean())
    for _, zr in ZONAS.iterrows():
        sub = am[(am.lap_distance >= zr.ini) & (am.lap_distance <= zr.fim)]
        tele["tel_vmin_" + zr.zona] = sub.groupby(CHAVES)["velocidade"].min()
    tele = tele.reset_index()
    TEL = [c for c in tele if c.startswith("tel_")]
    resumo_tele = tele.groupby(["carro", "setup_id"])[TEL].mean().reset_index()
    display(resumo_tele.round(1))

    vmin = resumo_tele.melt(id_vars=["carro", "setup_id"], value_vars=[c for c in TEL if c.startswith("tel_vmin")])
    px.line(vmin, x="variable", y="value", color="setup_id", facet_row="carro", markers=True,
            category_orders={"setup_id": SETUPS}, title="Velocidade mínima nas curvas lentas detectadas",
            labels={"variable": "curva", "value": "km/h"}).show()
    pedal = resumo_tele.melt(id_vars=["carro", "setup_id"], value_vars=["tel_pct_acel_cheio", "tel_pct_freando"])
    px.bar(pedal, x="setup_id", y="value", color="carro", barmode="group", facet_row="variable",
           category_orders={"setup_id": SETUPS}, title="% da volta com acelerador cheio / freando").show()

In [ ]:
if TEM_AMOSTRAS:
    # 6) ganho por trecho (20 trechos) vs. S0
    L = am["lap_distance"].quantile(.99)
    am["trecho"] = np.minimum(20, (am["lap_distance"] // (L / 20)).astype(int) + 1)
    tt = (am.groupby(CHAVES + ["trecho"])["tempo_volta_atual_ms"].agg(lambda s: (s.max() - s.min()) / 1000)
            .groupby(["carro", "setup_id", "trecho"]).mean().rename("t").reset_index())
    tt = tt.merge(tt[tt.setup_id == "S0"][["carro", "trecho", "t"]].rename(columns={"t": "t_s0"}), on=["carro", "trecho"])
    tt["delta"] = tt["t"] - tt["t_s0"]
    for carro, g in tt[tt.setup_id != "S0"].groupby("carro"):
        mat = g.pivot(index="setup_id", columns="trecho", values="delta").reindex([s for s in SETUPS if s != "S0"]).dropna(how="all")
        px.imshow(mat, color_continuous_scale="RdYlGn_r", color_continuous_midpoint=0, aspect="auto",
                  title=f"Ganho/perda por trecho vs. S0 (s) – {carro} (verde = mais rápido)").show()
    # 7) consistência
    cons = perfil.groupby(["carro", "setup_id"])["vel_dp"].mean().reset_index()
    px.bar(cons, x="setup_id", y="vel_dp", color="carro", barmode="group", category_orders={"setup_id": SETUPS},
           title="Consistência: dp da velocidade entre voltas (menor = mais repetível)", labels={"vel_dp": "km/h"}).show()

## 5. Features e divisão treino/teste
- **Alvo:** `tempo_s`. **Preditores (trilha A):** valores de setup **lidos do jogo** + `carro` + `volta_stint`. Não usamos o rótulo `setup_id` – com ele o modelo só decoraria médias e não conseguiria opinar sobre o S10.
- **S10 sai inteiro** (teste prático).
- **Teste:** 1 sessão inteira de cada carro (entre as sessões com setups S1–S9). Se algum carro tiver < 2 sessões, *fallback* `train_test_split` 75/25 estratificado por carro × setup.
- **Validação cruzada:** `GroupKFold` por **bloco**. Voltas vizinhas do mesmo bloco são quase iguais; separar só por volta vazaria informação. Usamos bloco (e não sessão) porque o treino tem 1 sessão por carro.
- **Pré-processamento:** one-hot do carro, `VarianceThreshold` (remove colunas constantes, = `step_zv`), `StandardScaler` (necessário para Lasso/Ridge).

**Perguntas:** *Por que essa proporção?* → 1 sessão por carro ≈ 40–50% das voltas; é o preço de testar em sessões nunca vistas. *Os setups do teste estão no treino?* → para o mesmo carro não: o modelo precisa generalizar pelos **valores** e pelo outro carro.

In [ ]:
PRED_A = PARAMS + ["carro", "volta_stint"]
modelagem = limpas[limpas.setup_id != "S10"].reset_index(drop=True)
s10 = limpas[limpas.setup_id == "S10"].reset_index(drop=True)

ses_exp = modelagem[modelagem.setup_id != "S0"][["carro", "sessao"]].drop_duplicates()
if (ses_exp.groupby("carro").size() >= 2).all():
    ses_teste = ses_exp.groupby("carro")["sessao"].max().tolist()
    TIPO_SPLIT = f"grupo: sessões {ses_teste} no teste"
    teste_mask = modelagem.sessao.isin(ses_teste)
else:
    TIPO_SPLIT = "fallback: train_test_split 75/25 estratificado por carro×setup"
    _, idx_te = train_test_split(modelagem.index, test_size=.25, random_state=SEMENTE,
                                 stratify=modelagem.carro + modelagem.setup_id)
    teste_mask = modelagem.index.isin(idx_te)
treino, teste = modelagem[~teste_mask].reset_index(drop=True), modelagem[teste_mask].reset_index(drop=True)
print(TIPO_SPLIT, "| treino:", len(treino), "| teste:", len(teste), "| S10:", len(s10))

# folds por bloco, guardados como mapa bloco -> fold para reusar IGUAL na trilha B
n_folds = min(5, treino.bloco.nunique())
BLOCO_FOLD = {}
for k, (_, te) in enumerate(GroupKFold(n_splits=n_folds).split(treino, groups=treino.bloco)):
    for b in treino.bloco.iloc[te].unique(): BLOCO_FOLD[b] = k
def folds_de(d):
    f = d.bloco.map(BLOCO_FOLD).values
    return [(np.where(f != k)[0], np.where(f == k)[0]) for k in range(n_folds)]
CV_A = folds_de(treino)

def preproc(num):
    return ColumnTransformer([("carro", OneHotEncoder(drop="first", handle_unknown="ignore"), ["carro"]),
                              ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("zv", VarianceThreshold(0.0)),
                                                ("esc", StandardScaler())]), num)])

## 6. Modelos – trilha A (previsão / deploy)
| Modelo | Papel |
|---|---|
| `DummyRegressor` | baseline: sempre a média. Todo modelo útil tem que ganhar dele. |
| Linear só com carro | responde **"é só o carro?"** |
| Lasso e Ridge | lineares, interpretáveis (Lasso zera parâmetros inúteis) |
| Random Forest | não linear, robusto |
| XGBoost (ou HistGradientBoosting) | boosting, costuma ser o melhor em tabular (o artigo GT3 usou LightGBM) |
| LightGBM (opcional) | mesma família do artigo GT3 |
| **Placebo** | Random Forest com os parâmetros de setup **embaralhados** – se empatar com o real, o "efeito do setup" é ilusão |

Tuning com `GridSearchCV` nas **mesmas folds por bloco**, escolhendo pelo **MAE**. Métricas: MAE (erro médio em s), RMSE (pune erros grandes), R².

**Perguntas:** *Por que 3+ modelos?* · *Para que serve o placebo?* · *Por que escolher por MAE?* (é em segundos, fácil de explicar).

In [ ]:
SCORING = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}
def resumo_cv(cvres, i=None):
    if i is None:   # cross_validate
        return {m: (-1 if m != "r2" else 1) * np.mean(cvres[f"test_{m}"]) for m in SCORING}
    return {m: (-1 if m != "r2" else 1) * cvres[f"mean_test_{m}"][i] for m in SCORING}

def espaco_modelos(num):
    pre = lambda: preproc(num)
    esp = {
        "lasso": (Pipeline([("pre", pre()), ("m", Lasso(max_iter=20000))]), {"m__alpha": np.logspace(-4, 0, 12)}),
        "ridge": (Pipeline([("pre", pre()), ("m", Ridge())]), {"m__alpha": np.logspace(-3, 3, 12)}),
        "random_forest": (Pipeline([("pre", pre()), ("m", RandomForestRegressor(n_estimators=300, random_state=SEMENTE, n_jobs=1))]),
                          {"m__max_features": [0.3, 0.6, 1.0], "m__min_samples_leaf": [2, 5, 10]}),
    }
    if TEM_XGB:
        esp["xgboost"] = (Pipeline([("pre", pre()), ("m", XGBRegressor(n_estimators=300, random_state=SEMENTE, n_jobs=1, verbosity=0))]),
                          {"m__max_depth": [2, 4, 6], "m__learning_rate": [0.03, 0.1], "m__min_child_weight": [1, 5]})
    else:
        esp["hist_gb"] = (Pipeline([("pre", pre()), ("m", HistGradientBoostingRegressor(random_state=SEMENTE))]),
                          {"m__max_depth": [2, 4, None], "m__learning_rate": [0.03, 0.1]})
    if TEM_LGBM:
        esp["lightgbm"] = (Pipeline([("pre", pre()), ("m", LGBMRegressor(n_estimators=300, random_state=SEMENTE, verbose=-1, n_jobs=1))]),
                           {"m__num_leaves": [7, 15], "m__learning_rate": [0.03, 0.1], "m__min_child_samples": [5, 20]})
    return esp

def rodar(X, y, cv, num, extra=True):
    res, busca = {}, {}
    if extra:
        res["baseline_media"] = resumo_cv(cross_validate(DummyRegressor(), X, y, cv=cv, scoring=SCORING))
        so_carro = Pipeline([("pre", ColumnTransformer([("c", OneHotEncoder(drop="first", handle_unknown="ignore"), ["carro"])])),
                             ("m", LinearRegression())])
        res["so_carro"] = resumo_cv(cross_validate(so_carro, X, y, cv=cv, scoring=SCORING))
    for nome, (pipe, grade) in espaco_modelos(num).items():
        gs = GridSearchCV(pipe, grade, cv=cv, scoring=SCORING, refit="mae", n_jobs=-1).fit(X, y)
        busca[nome] = gs; res[nome] = resumo_cv(gs.cv_results_, gs.best_index_)
    return pd.DataFrame(res).T, busca

X_tr, y_tr = treino[PRED_A], treino["tempo_s"]
tab_A, busca_A = rodar(X_tr, y_tr, CV_A, PARAMS + ["volta_stint"])

# placebo: parâmetros de setup embaralhados JUNTOS (carro e volta ficam no lugar)
rng = np.random.default_rng(7)
X_plac = X_tr.copy(); X_plac[PARAMS] = X_tr[PARAMS].values[rng.permutation(len(X_tr))]
plac = Pipeline([("pre", preproc(PARAMS + ["volta_stint"])), ("m", RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=SEMENTE, n_jobs=1))])
tab_A.loc["placebo_setup_embaralhado"] = resumo_cv(cross_validate(plac, X_plac, y_tr, cv=CV_A, scoring=SCORING))
tab_A = tab_A.sort_values("mae")
display(tab_A.round(3))

### 6a. Comparação e escolha do melhor
**Por quê:** escolhemos pela **validação cruzada** (o teste fica guardado para o fim). O vencedor precisa ganhar do baseline, do só-carro **e** do placebo; se empatar com o Lasso, vale defender o Lasso por ser mais simples.

**Perguntas:** *Por que esse modelo?* · *Quanto ele ganha do só-carro?* (isso responde "carro ou setup?").

In [ ]:
t = tab_A.reset_index(names="modelo").melt(id_vars="modelo", var_name="métrica", value_name="valor")
px.bar(t, x="valor", y="modelo", color="modelo", facet_col="métrica", orientation="h",
       title="Trilha A – validação cruzada por bloco").update_xaxes(matches=None).update_layout(showlegend=False).show()
MELHOR = tab_A.loc[list(busca_A)].mae.idxmin()
print("Melhor (menor MAE na CV):", MELHOR, "| parâmetros:", busca_A[MELHOR].best_params_)
print(f"Ganho de MAE vs só-carro: {tab_A.loc['so_carro','mae']-tab_A.loc[MELHOR,'mae']:.3f} s | "
      f"vs placebo: {tab_A.loc['placebo_setup_embaralhado','mae']-tab_A.loc[MELHOR,'mae']:.3f} s")

## 7. Teste final (sessões nunca vistas)
Treinamos o melhor no **treino inteiro** e avaliamos **uma vez** no teste, junto com baseline e só-carro. Depois: previsto × real, **importância por permutação** (quanto o MAE piora ao embaralhar cada variável) e **SHAP** (contribuição de cada variável em cada volta), se disponível.

**Perguntas:** *O erro no teste parece com o da CV?* (se for muito pior: overfitting ou sessão diferente) · *Qual parâmetro mais pesa? Faz sentido físico?* · *Importância é causa?* (não).

In [ ]:
modelo_A = busca_A[MELHOR].best_estimator_            # já reajustado no treino inteiro (refit)
X_te, y_te = teste[PRED_A], teste["tempo_s"]
def metr(y, p): return {"mae": mean_absolute_error(y, p), "rmse": root_mean_squared_error(y, p), "r2": r2_score(y, p)}
so_carro = Pipeline([("pre", ColumnTransformer([("c", OneHotEncoder(drop="first", handle_unknown="ignore"), ["carro"])])), ("m", LinearRegression())]).fit(X_tr, y_tr)
teste_tab = pd.DataFrame({MELHOR: metr(y_te, modelo_A.predict(X_te)),
                          "so_carro": metr(y_te, so_carro.predict(X_te)),
                          "baseline_media": metr(y_te, np.full(len(y_te), y_tr.mean()))}).T
display(teste_tab.round(3))

pv = teste.assign(previsto=modelo_A.predict(X_te))
fig = px.scatter(pv, x="tempo_s", y="previsto", color="carro", hover_data=["setup_id", "sessao", "volta"],
                 title=f"Previsto × real – teste – {MELHOR}")
lim = [pv[["tempo_s", "previsto"]].min().min(), pv[["tempo_s", "previsto"]].max().max()]
fig.add_shape(type="line", x0=lim[0], y0=lim[0], x1=lim[1], y1=lim[1], line=dict(dash="dash")); fig.show()

pi = permutation_importance(modelo_A, X_te, y_te, scoring="neg_mean_absolute_error", n_repeats=20, random_state=SEMENTE)
imp = pd.DataFrame({"variavel": PRED_A, "aumento_MAE_s": pi.importances_mean, "dp": pi.importances_std}).sort_values("aumento_MAE_s")
px.bar(imp, x="aumento_MAE_s", y="variavel", error_x="dp", orientation="h",
       title="Importância por permutação (quanto o MAE piora, em s)").show()

In [ ]:
if TEM_SHAP:
    try:
        import shap
        pre, est = modelo_A.named_steps["pre"], modelo_A.named_steps["m"]
        Xt = pre.transform(X_tr); nomes = pre.get_feature_names_out()
        expl = shap.Explainer(est, Xt, feature_names=nomes) if not isinstance(est, (Lasso, Ridge)) else shap.LinearExplainer(est, Xt, feature_names=nomes)
        sv = expl(Xt[:300])
        shap.plots.beeswarm(sv, max_display=15, show=False); plt.title("SHAP – trilha A"); plt.show()
    except Exception as e:
        print("SHAP não rodou para este modelo:", e)
else:
    print("shap não instalado – use a importância por permutação acima")

## 8. Trilha B – modelo **explicativo** (setup + carro + telemetria)
| | Trilha A – previsão (deploy) | Trilha B – explicativa |
|---|---|---|
| Entradas | setup + carro + nº da volta | A + agregados de telemetria (vel. mín. nas curvas, vel. máx., % acelerador, % freio, pneus, g lateral) |
| Pergunta | "com este setup, que tempo espero?" | "quanto do tempo vem da forma de pilotar? o que pesa?" |
| Vai para o app? | **Sim** | **Não** – telemetria só existe depois da volta (vazamento) |

**Mesmas voltas, mesmas folds (mesmo bloco → mesmo fold), mesmos modelos.** Ficam **fora** de B: tempos por trecho e **velocidade média** (distância ÷ tempo = o próprio alvo disfarçado).

**Perguntas:** *Por que não usar B no app se ele erra menos?* · *Por que tirar a velocidade média?* · *Velocidade mínima na curva é causa ou consequência?* (é o mecanismo pelo qual setup e piloto ganham tempo).

**Extensão (não implementada):** trilha C por trecho, como no artigo GT3 – prever o tempo de cada um dos 20 trechos e somar.

In [ ]:
if TEM_AMOSTRAS:
    TEL_B = [c for c in TEL if c != "tel_vel_media"]
    trB, teB = treino.merge(tele, on=CHAVES), teste.merge(tele, on=CHAVES)
    PRED_B = PRED_A + TEL_B
    tab_B, busca_B = rodar(trB[PRED_B], trB["tempo_s"], folds_de(trB), PARAMS + ["volta_stint"] + TEL_B, extra=False)
    comp = pd.concat([tab_A.loc[list(busca_A)].assign(trilha="A (previsão)"), tab_B.assign(trilha="B (explicativa)")]).reset_index(names="modelo")
    display(comp.round(3))
    px.bar(comp, x="modelo", y="mae", color="trilha", barmode="group", title="MAE na CV: trilha A × trilha B").show()

    MELHOR_B = tab_B.mae.idxmin(); modelo_B = busca_B[MELHOR_B].best_estimator_
    display(pd.DataFrame({"A (" + MELHOR + ")": metr(y_te, modelo_A.predict(X_te)),
                          "B (" + MELHOR_B + ")": metr(teB["tempo_s"], modelo_B.predict(teB[PRED_B]))}).T.round(3))
    piB = permutation_importance(modelo_B, teB[PRED_B], teB["tempo_s"], scoring="neg_mean_absolute_error", n_repeats=10, random_state=SEMENTE)
    impB = pd.DataFrame({"variavel": PRED_B, "aumento_MAE_s": piB.importances_mean})
    impB["tipo"] = np.where(impB.variavel.str.startswith("tel_"), "telemetria (pilotagem)", "setup/carro")
    px.bar(impB.sort_values("aumento_MAE_s").tail(15), x="aumento_MAE_s", y="variavel", color="tipo", orientation="h",
           title=f"Trilha B – o que mais explica o tempo ({MELHOR_B})").show()
    print("AVISO: o modelo B NÃO vai para o deploy – usa dados que só existem depois da volta.")
else:
    print("Sem amostras.csv – trilha B pulada.")

## 9. Ranking de TODOS os setups (S0–S10) por carro + teste prático do S10
Para cada carro: média real, n, dp, diferença vs. S0, **teste de Welch** vs. S0 e **veredito**: "mais rápido/lento" só se |diferença| > **ruído do piloto** **e** p < 0,05; senão "empate".
Previsão do modelo A para cada setup, com a origem marcada:
- `CV (fora da dobra)`: voltas do treino previstas por um modelo que não as viu;
- `teste`: sessões de teste;
- `S10 (nunca visto)`.
`no_treino` diz se aquele setup (naquele carro) estava no treino.

**Alerta de extrapolação:** o S10 usa valores fora da faixa testada (suspensão 12/7, barra traseira 2, altura 36/42…). Árvores não extrapolam – fora da faixa "congelam" no último valor visto.

**Perguntas:** *Qual o melhor setup de cada carro?* · *O S10 do GPT é mesmo mais rápido?* · *Por que Welch?* (variâncias diferentes) · *10 testes não dão falso positivo?* (por isso a régua do ruído; dá para citar Bonferroni) · *Por que o S10 não diz qual ajuste ajudou?* (mudou vários de uma vez).

In [ ]:
# extrapolação do S10
if len(s10):
    faixa = modelagem[PARAMS].agg(["min", "max"]).T
    faixa["S10"] = s10[PARAMS].mean()
    fora = faixa[(faixa.S10 < faixa["min"]) | (faixa.S10 > faixa["max"])]
    if len(fora): print("ATENÇÃO – S10 fora da faixa de treino (extrapolação):"); display(fora)

# previsões: fora da dobra (treino), teste e S10
oof = cross_val_predict(busca_A[MELHOR].estimator.set_params(**busca_A[MELHOR].best_params_), X_tr, y_tr, cv=CV_A)
prev = pd.concat([treino.assign(previsto=oof, origem="CV (fora da dobra)"),
                  teste.assign(previsto=modelo_A.predict(X_te), origem="teste")]
                 + ([s10.assign(previsto=modelo_A.predict(s10[PRED_A]), origem="S10 (nunca visto)")] if len(s10) else []))
p_setup = prev.groupby(["carro", "setup_id"]).agg(previsto_medio=("previsto", "mean"),
                                                 origem_previsao=("origem", lambda s: " + ".join(sorted(set(s))))).reset_index()
no_treino = set(map(tuple, treino[["carro", "setup_id"]].drop_duplicates().values))

linhas = []
for (carro, st), g in limpas.groupby(["carro", "setup_id"]):
    s0 = limpas[(limpas.carro == carro) & (limpas.setup_id == "S0")]["tempo_s"]
    p = np.nan if st == "S0" or len(g) < 2 else stats.ttest_ind(g.tempo_s, s0, equal_var=False).pvalue
    dif = g.tempo_s.mean() - s0.mean()
    ver = ("referência" if st == "S0" else "mais rápido" if dif < -RUIDO and p < .05
           else "mais lento" if dif > RUIDO and p < .05 else "empate (dentro do ruído)")
    linhas.append(dict(carro=carro, setup_id=st, n=len(g), media_real=g.tempo_s.mean(), dp=g.tempo_s.std(),
                       dif_vs_s0=dif, p_welch=p, veredito=ver, no_treino=(carro, st) in no_treino))
ranking = (pd.DataFrame(linhas).merge(p_setup, on=["carro", "setup_id"], how="left")
             .sort_values(["carro", "media_real"]))
ranking["posicao"] = ranking.groupby("carro").cumcount() + 1
ranking["erro_previsao"] = ranking.previsto_medio - ranking.media_real
print(f"Régua do ruído do piloto: ±{RUIDO:.3f} s")
for carro, g in ranking.groupby("carro"):
    print("\n==== Ranking –", carro, "===="); display(g.drop(columns="carro").round(3))
ranking.to_csv(os.path.join(PASTA, "ranking_setups_python.csv"), index=False)

In [ ]:
CORES = {"referência": "royalblue", "mais rápido": "green", "mais lento": "firebrick", "empate (dentro do ruído)": "grey"}
for carro, g in ranking.groupby("carro"):
    g = g.sort_values("media_real", ascending=False); s0m = g.loc[g.setup_id == "S0", "media_real"].iloc[0]
    fig = go.Figure()
    fig.add_vrect(x0=s0m - RUIDO, x1=s0m + RUIDO, fillcolor="lightgrey", opacity=.4, line_width=0, annotation_text="ruído do piloto")
    fig.add_vline(x=s0m, line_dash="dash", annotation_text="S0")
    for ver, gg in g.groupby("veredito"):
        fig.add_trace(go.Scatter(x=gg.media_real, y=gg.setup_id, mode="markers", name=ver, marker=dict(size=12, color=CORES[ver]),
                                 error_x=dict(type="data", array=gg.dp), hovertext=gg.origem_previsao))
    fig.add_trace(go.Scatter(x=g.previsto_medio, y=g.setup_id, mode="markers", name="previsto (modelo A)",
                             marker=dict(symbol="x", size=11, color="black")))
    fig.update_layout(title=f"Ranking de setups – {carro} (ponto = real ± dp, X = previsto)", xaxis_title="Tempo médio (s)",
                      yaxis=dict(categoryorder="array", categoryarray=list(g.setup_id)))
    fig.show()

if len(s10):
    s10_tab = ranking[ranking.setup_id == "S10"][["carro", "n", "media_real", "previsto_medio", "dif_vs_s0", "p_welch", "veredito"]]
    print("\nTESTE PRÁTICO – S10 (setup do GPT):"); display(s10_tab.round(3))
else:
    print("Ainda não há voltas do S10 – grave as sessões 5 (Ferrari) e 6 (Alfa Romeo).")

## 10. Salvar o modelo para o deploy
Salvamos o **pipeline completo** da trilha A (pré-processamento + modelo) com `joblib`, mais um JSON com preditores, faixas de treino (para o app avisar extrapolação), ruído do piloto e métricas.
No app (Streamlit/Gradio/Flask): `m = joblib.load("modelo_final_python.joblib"); m.predict(pd.DataFrame([entrada]))`.

**Perguntas:** *Por que salvar o pipeline e não só o modelo?* → a padronização e o one-hot fazem parte do modelo; sem eles a previsão sai errada. *Por que o modelo B não foi salvo?* → vazamento.

In [ ]:
joblib.dump(modelo_A, os.path.join(PASTA, "modelo_final_python.joblib"))
info = {"modelo": MELHOR, "preditores": PRED_A, "params": PARAMS, "ruido_piloto_s": RUIDO,
        "faixas": {c: [float(modelagem[c].min()), float(modelagem[c].max())] for c in PARAMS},
        "metricas_teste": {k: float(v) for k, v in metr(y_te, modelo_A.predict(X_te)).items()},
        "divisao": TIPO_SPLIT, "sintetico": USAR_DADOS_SINTETICOS}
json.dump(info, open(os.path.join(PASTA, "modelo_info_python.json"), "w"), ensure_ascii=False, indent=1)
print("Salvo:", os.path.join(PASTA, "modelo_final_python.joblib"), "+ modelo_info_python.json")

In [ ]:
# === Exportar para a LANDING PAGE e a API (landing/data/resultados.json) ===
import sklearn
info["sklearn_versao"] = sklearn.__version__   # a API precisa da MESMA versão no requirements.txt
json.dump(info, open(os.path.join(PASTA, "modelo_info_python.json"), "w"), ensure_ascii=False, indent=1)
X_te_, y_te_ = teste[PRED_A], teste["tempo_s"]
comp_land = []
for nome in tab_A.index:
    cvm = {k: float(tab_A.loc[nome, k]) for k in ["mae", "rmse", "r2"]}
    t = None
    if nome in busca_A: t = metr(y_te_, busca_A[nome].best_estimator_.predict(X_te_))
    elif nome in teste_tab.index: t = {k: float(teste_tab.loc[nome, k]) for k in ["mae", "rmse", "r2"]}
    comp_land.append({"modelo": nome, "cv": cvm, "teste": {k: float(v) for k, v in t.items()} if t else None})
res_land = {"exemplo": bool(USAR_DADOS_SINTETICOS), "aviso": "dados sintéticos" if USAR_DADOS_SINTETICOS else "",
            "melhor_modelo": MELHOR, "ruido_piloto_s": float(RUIDO), "comparacao_modelos": comp_land,
            "importancia": [{"variavel": r.variavel, "aumento_mae_s": float(r.aumento_MAE_s)} for r in imp.sort_values("aumento_MAE_s", ascending=False).itertuples()],
            "ranking": [{k: (None if pd.isna(v) else (v.item() if hasattr(v, "item") else v)) for k, v in r.items()}
                        for r in ranking[["carro", "setup_id", "n", "media_real", "dif_vs_s0", "previsto_medio", "veredito", "p_welch"]].to_dict("records")]}
json.dump(res_land, open(os.path.join(PASTA, "resultados.json"), "w"), ensure_ascii=False, indent=1)
print("Copie resultados.json para landing/data/ e os 2 arquivos do modelo para api/modelo/")


## 10. Registro de modelos (para a API e o site comparar VÁRIOS modelos)
Exporta **todos** os algoritmos (não só o melhor) em `modelos/<id>/modelo.joblib` + `meta.json`, e um índice `modelos/registro.json` com o modelo **padrão**.

- `meta.json`: nome, algoritmo, hiperparâmetros resumidos (`get_params`), métricas da **CV agrupada** (MAE/RMSE/R² com desvio-padrão entre dobras), métricas de **teste**, colunas, alvo, `sklearn_versao`, data e `sintetico`.
- **Segundo alvo – velocidade:** além de `tempo_s`, treinamos modelos para `vel_max_kmh` (velocidade máxima da volta, normalmente no túnel) e `vel_media_kmh` (velocidade média da volta), colunas **reais** do `voltas.csv` gravadas pelo `capturar_telemetria.py`.
- **Sem vazamento:** as entradas continuam **só setup + carro + volta_stint** — valores conhecidos *antes* da volta. Nenhuma coluna medida durante a volta (telemetria, tempo, outras velocidades) entra no X. Por isso esses modelos podem ser usados no simulador.
- Observação: `vel_media_kmh` é quase o inverso do tempo de volta (distância fixa), então traz pouca informação nova; `vel_max_kmh` é mais interessante porque reage à asa traseira (arrasto).
- O padrão de cada alvo é o menor MAE de CV entre os modelos “de verdade” (baseline e só-carro ficam como referência). A API permite trocar com a variável `MODELO_PADRAO`.
- Mesmo código do arquivo `api/registro_modelos.py` do projeto.


In [ ]:
# === Registro de modelos: modelos/<id>/{modelo.joblib, meta.json} + modelos/registro.json ===
"""Treina e exporta TODOS os modelos para um registro desacoplado:
    modelos/<id>/modelo.joblib + modelos/<id>/meta.json  e  modelos/registro.json
Alvos: tempo_s (tempo de volta) e, se existirem no voltas.csv, vel_max_kmh e vel_media_kmh.
Entrada SEMPRE só setup + carro (+ volta_stint) -> sem vazamento: nada medido durante a volta entra no X.
Este mesmo código está colado na célula "Registro de modelos" do notebook."""
import json, os, re, shutil, time, datetime
import numpy as np, pandas as pd, joblib, sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ALVOS = {"tempo_s": ("tempo de volta", "s"), "vel_max_kmh": ("velocidade máxima da volta", "km/h"),
         "vel_media_kmh": ("velocidade média da volta", "km/h")}

def _pre(num):
    return ColumnTransformer([("carro", OneHotEncoder(drop="first", handle_unknown="ignore"), ["carro"]),
                              ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num)])

def espaco(num, semente=42):
    e = {"baseline_media": ("Dummy (média)", DummyRegressor(), {}),
         "so_carro": ("Regressão linear só com o carro",
                      Pipeline([("pre", ColumnTransformer([("c", OneHotEncoder(drop="first", handle_unknown="ignore"), ["carro"])])), ("m", LinearRegression())]), {}),
         "lasso": ("Lasso", Pipeline([("pre", _pre(num)), ("m", Lasso(max_iter=20000))]), {"m__alpha": list(np.logspace(-4, 0, 8))}),
         "ridge": ("Ridge", Pipeline([("pre", _pre(num)), ("m", Ridge())]), {"m__alpha": list(np.logspace(-3, 3, 8))}),
         "random_forest": ("Random Forest", Pipeline([("pre", _pre(num)), ("m", RandomForestRegressor(n_estimators=200, random_state=semente, n_jobs=1))]),
                           {"m__max_features": [0.3, 0.6, 1.0], "m__min_samples_leaf": [2, 5, 10]})}
    try:
        from xgboost import XGBRegressor
        e["xgboost"] = ("XGBoost", Pipeline([("pre", _pre(num)), ("m", XGBRegressor(n_estimators=200, random_state=semente, n_jobs=1, verbosity=0))]),
                        {"m__max_depth": [2, 4], "m__learning_rate": [0.03, 0.1], "m__min_child_weight": [1, 5]})
    except ImportError: pass
    try:
        from lightgbm import LGBMRegressor
        e["lightgbm"] = ("LightGBM", Pipeline([("pre", _pre(num)), ("m", LGBMRegressor(n_estimators=200, random_state=semente, verbose=-1, n_jobs=1))]),
                         {"m__num_leaves": [7, 15], "m__learning_rate": [0.03, 0.1], "m__min_child_samples": [5, 20]})
    except ImportError: pass
    return e

def _limpo(v):
    if isinstance(v, (np.floating, float)): return float(v)
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (int, str, bool)) or v is None: return v
    return re.sub(r"\s+", " ", str(v))[:80]

def resumo_params(est):
    m = est.named_steps["m"] if hasattr(est, "named_steps") else est
    p = m.get_params()
    chaves = ["alpha", "n_estimators", "max_depth", "max_features", "min_samples_leaf", "learning_rate",
              "min_child_weight", "num_leaves", "min_child_samples", "strategy", "fit_intercept"]
    return {k: _limpo(p[k]) for k in chaves if k in p and p[k] is not None}

def metr(y, p):
    return {"mae": float(mean_absolute_error(y, p)), "rmse": float(root_mean_squared_error(y, p)), "r2": float(r2_score(y, p))}

def exportar(treino, teste, cv, params, saida, sintetico, extra=("volta_stint",), alvos=None, log=print, ruido=None):
    """treino/teste: DataFrames já limpos (com carro, setup e alvos). cv: lista de (idx_treino, idx_val) no treino."""
    pred = list(params) + ["carro"] + [c for c in extra if c in treino]
    num = list(params) + [c for c in extra if c in treino]
    alvos = [a for a in (alvos or ALVOS) if a in treino and treino[a].notna().sum() > 10 and treino[a].std() > 0]
    if os.path.isdir(saida): shutil.rmtree(saida)
    os.makedirs(saida)
    sc = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}
    reg = {"gerado_em": datetime.datetime.now().isoformat(timespec="seconds"), "sintetico": bool(sintetico),
           "sklearn_versao": sklearn.__version__, "ruido_piloto_s": None if ruido is None else float(ruido), "alvos": {}, "modelos": []}
    for alvo in alvos:
        ok_tr, ok_te = treino[alvo].notna().values, teste[alvo].notna().values
        Xtr, ytr = treino.loc[ok_tr, pred], treino.loc[ok_tr, alvo]
        Xte, yte = teste.loc[ok_te, pred], teste.loc[ok_te, alvo]
        pos = np.where(ok_tr)[0]; mapa = {p: i for i, p in enumerate(pos)}
        cv_a = [(np.array([mapa[i] for i in a if i in mapa]), np.array([mapa[i] for i in b if i in mapa])) for a, b in cv]
        ids = []
        for nome, (rotulo, pipe, grade) in espaco(num).items():
            t0 = time.time()
            est = GridSearchCV(pipe, grade, cv=cv_a, scoring="neg_mean_absolute_error", n_jobs=-1).fit(Xtr, ytr).best_estimator_ if grade else pipe.fit(Xtr, ytr)
            r = cross_validate(est, Xtr, ytr, cv=cv_a, scoring=sc)
            cvm = {k: float((-1 if k != "r2" else 1) * np.mean(r[f"test_{k}"])) for k in sc}
            cvm.update({k + "_dp": float(np.std(r[f"test_{k}"])) for k in sc})
            est.fit(Xtr, ytr)
            mid = f"{'tempo' if alvo == 'tempo_s' else alvo.replace('_kmh', '')}__{nome}"
            os.makedirs(os.path.join(saida, mid))
            joblib.dump(est, os.path.join(saida, mid, "modelo.joblib"))
            meta = {"id": mid, "nome": rotulo, "algoritmo": nome, "alvo": alvo, "alvo_desc": ALVOS[alvo][0], "unidade": ALVOS[alvo][1],
                    "hiperparametros": resumo_params(est), "metricas_cv": cvm,
                    "metricas_teste": metr(yte, est.predict(Xte)) if len(yte) else None,
                    "colunas": pred, "params_setup": list(params),
                    "faixas": {c: [float(treino[c].min()), float(treino[c].max())] for c in params},
                    "n_treino": int(len(ytr)), "n_teste": int(len(yte)), "sklearn_versao": sklearn.__version__,
                    "data": reg["gerado_em"], "sintetico": bool(sintetico), "treino_s": round(time.time() - t0, 2)}
            json.dump(meta, open(os.path.join(saida, mid, "meta.json"), "w"), ensure_ascii=False, indent=1)
            reg["modelos"].append({k: meta[k] for k in ["id", "nome", "algoritmo", "alvo", "unidade", "metricas_cv", "metricas_teste", "hiperparametros"]})
            ids.append((cvm["mae"], nome, mid)); log(f"{mid:32s} CV MAE {cvm['mae']:.3f} ± {cvm['mae_dp']:.3f}")
        cand = [x for x in ids if x[1] not in ("baseline_media", "so_carro")] or ids
        reg["alvos"][alvo] = {"descricao": ALVOS[alvo][0], "unidade": ALVOS[alvo][1], "padrao": min(cand)[2]}
    reg["padrao"] = reg["alvos"].get("tempo_s", {}).get("padrao")
    json.dump(reg, open(os.path.join(saida, "registro.json"), "w"), ensure_ascii=False, indent=1)
    log(f"Registro: {os.path.join(saida, 'registro.json')} | padrão tempo = {reg['padrao']}")
    return reg


PASTA_MODELOS = os.path.join(PASTA, "modelos")
registro = exportar(treino, teste, list(CV_A), PARAMS, PASTA_MODELOS, sintetico=USAR_DADOS_SINTETICOS, ruido=RUIDO)
display(pd.DataFrame([{"id": m["id"], "alvo": m["alvo"], "CV MAE": m["metricas_cv"]["mae"], "± dp": m["metricas_cv"]["mae_dp"],
                       "teste MAE": (m["metricas_teste"] or {}).get("mae")} for m in registro["modelos"]]).round(3))
import shutil
shutil.make_archive(os.path.join(PASTA, "modelos"), "zip", PASTA_MODELOS)
print("Baixe modelos.zip e descompacte em <projeto>/modelos/ (e em api/modelos/ para o deploy).")
try:
    from google.colab import files; files.download(os.path.join(PASTA, "modelos.zip"))
except Exception: pass
